# Orbit Wars 2026
太陽の周りを公転する惑星を征服せよ！プレイヤーは惑星間を艦隊で移動し、100×100の連続した空間内で領土を占領する。

### ゲームメカニズム
- 惑星は毎ターン（半径に比例して）宇宙船を生成する。
- 内惑星は中心の太陽の周りを公転し、外惑星は静止している。
- 艦隊は出発惑星から一定の角度で直線的に飛行する。
- 艦隊の速度は艦隊の規模に応じて変化する（艦船1隻＝1ターンあたり1回、大型艦隊では最大6ターンあたり1回まで）。
- 戦闘：到着した艦隊艦艇は惑星の駐屯兵力から差し引かれる。駐屯兵力が0を下回ると、所有権が反転する。
- 太陽：太陽に衝突した艦隊は破壊される
- 彗星：楕円軌道を描いて盤面を通過する一時的な惑星
<br>
勝利条件：制限時間終了時、またはプレイヤーが1人だけになった時点で、最も多くの艦船（惑星＋艦隊）を保有しているプレイヤー

In [1]:
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 21.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_clobber
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_coin_game
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_go
[kaggle_environments.envs.open_s

In [ ]:
from sklearn.utils import extmath
BOARD = 100.0
CENTER_X, CENTER_Y = 50.0, 50.0
SUN_R = 10.0
MAX_SPEED = 6.0
SUN_SAFETY = 1.5
ROTATION_LIMIT = 50.0
TOTAL_STEPS = 500

# 距離計算
def dist(ax, ay, bx, by):
    return math.hypot(ax - bx, ay - by)

# 艦隊の速度
def fleet_speed(ships):
    if ships <= 1:
        return 1.0
    ratio = math.log(ships) / math.log(1000.0)
    ratio = max(0.0, min(1.0, ratio))   # 1 <= ratio <= 0
    return 1.0 + (MAX_SPEED - 1.0) * (ratio ** 1.5)

# セグメントが太陽に当たるかの判定
def segment_hits_sun(x1, y1, x2, y2, safety=SUN_SAFETY):
    r = SUN_R + safety
    dx, dy = x2 - x1, y2 - y1
    fx, fy = x1 - CENTER_X, y1 - CENTER_Y
    a = dx * dx + dy * dy # 二次項の係数
    if a < 1e-9: # ゼロ除算の回避
        return dist(x1, y1, CENTER_X, CENTER_Y) < r
    b = 2 * (fx * dx + fy * dy) # 二次方程式の係数の計算
    c = fx * fx + fy * fy - r * r # 
    disc = b * b - 4 * a * c # 判別式
    if disc < 0:
        return False
    disc = math.sqrt(disc)
    t1 = (-b - disc) / (2 * a)
    t2 = (-b + disc) / (2 * a)
    return (0 <= t1 <= 1) or (0 <= t2 <= 1)

# 安全な角度と距離の計算
def safe_angle_and_distance(sx, sy, tx, ty):
    if not segment_hits_sun(sx, sy, tx, ty): # 起動が太陽と衝突するかの判定
        return math.atan2(ty - sy, tx - sx), dist(sx, sy, tx, ty)
    vx, vy = tx - sx, ty - sy
    norm = math.hypot(vx, vy)
    if norm < 1e-9:
        return math.atan2(ty - sy, tx - sx), dist(sx, sy, tx, ty) # 正規化
    nx, ny = -vy / norm, vx / norm # 進行方向に対する法線ベクトルの演算
    best = None
    for sign in (1.0, -1.0): # 迂回路の探索ループ
        for mult in (2.0, 3.0, 4.0):
            wx = CENTER_X + sign * nx * (SUN_R * mult)
            wy = CENTER_Y + sign * ny * (SUN_R * mult)
            if segment_hits_sun(sx, sy, wx, wy) or segment_hits_sun(wx, wy, tx, ty):
                continue
            d = dist(sx, sy, wx, wy) + dist(wx, wy, tx, ty)
            if best is None or d < best[0]:
                best = (d, wx, wy)
            break
    if best is None:
        ang = math.atan2(ty - sy, tx - sx)
        return ang, dist(sx, sy, tx, ty) * 1.5
    _, wx, wy = best
    ang = math.atan2(wy - sy, wx - sx)
    return ang, best[0]

def predict_platnet_position(planet, initial_by_id, angular_velocity, turns):
    init = initial_by_id.get(planet.id)
    if init is None:
        return planet.x, planet.y1
    orbital_r = dist(init.x, init.y, CENTER_X, CENTER_Y)
    if orbital_r + init.radius >= ROTATION_LIMIT:
        return planet.x, planet.y
    cur_ang = math.atan2(planet.y - CENTER_Y, planet.x - CENTER_X)
    new_ang = cur_ang + angular_velocity * turns
    return (CENTER_X + orbital_r * math.cos(new_ang),
            CENTER_Y + orbital_r * math.sin(new_ang))

def predict_comet_position(planet_id, comets, turns):
    for g in comets:
        pids = g.get("planet_ids", [])
        if planet_id not in pids:
            continue
        idx = pids.index(planet_id)
        paths = g.get("paths", [])
        path_index = g.get("path_index", 0)
        if idx >= len(paths):
            return None
        path = paths[idx]
        future_idx = path_index + int(turns)
        if 0 <= future_idx < len(path):
            return path[future_idx][0], path[future_idx][1]
        return None
    return None

def comet_remaining_life(planet_id, comets):
    for g in comets:
        pids = g.get("planet_ids", [])
        if planet_id not in pids:
            continue
        idx = pids.index(planet_id)
        paths = g.get("paths", [])
        path_index = g.get("path_index", 0)
        if idx < len(paths):
            return max(0, len(paths[idx]) - path_index)
    return 0

def estimate_arrival(sx, sy, tx, ty, ships):
    angle, d = safe_angle_and_distance(sx, sy, tx, ty)
    return angle, max(1, int(math.ceil(d / fleet_speed(ships))))